In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# Load dataset
df = pd.read_csv(r"C:\Users\Keertu\Downloads\Churn_Modelling.csv")

# Create a copy for processing
churn = df.copy()

# Remove unnecessary identifier columns
churn.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)

# Remove duplicate records if any exist
churn.drop_duplicates(inplace=True)

# Handle missing values (if present in specific columns)
churn["Balance"] = churn["Balance"].fillna(0)
churn["Geography"] = churn["Geography"].fillna("Unknown")

# Create derived analytical columns / business bins
# 1. Credit score tiers
churn["credit_score_segment"] = pd.cut(
    churn["CreditScore"],
    bins=[300, 580, 670, 740, 800, 850],
    labels=["Poor", "Fair", "Good", "Very Good", "Exceptional"]
)

# 2. Age group categories
churn["age_group"] = pd.cut(
    churn["Age"],
    bins=[17, 30, 45, 60, 100],
    labels=["Young Adult", "Middle-Aged", "Mature Adult", "Senior"]
)

# 3. Balance-to-estimated salary ratio
churn["balance_salary_ratio"] = (churn["Balance"] / (churn["EstimatedSalary"] + 1)).round(4)

# 4. Active engagement label
churn["engagement_status"] = churn["IsActiveMember"].map({1: "Active", 0: "Inactive"})

# 5. Churn label for readable visualization
churn["churn_label"] = churn["Exited"].map({1: "Churned", 0: "Retained"})

# Standardize column names (lowercase, no spaces, underscores)
churn.columns = (
    churn.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

print("Original Shape:", df.shape)
print("Cleaned Shape:", churn.shape)
churn.head()

Original Shape: (10000, 14)
Cleaned Shape: (10000, 16)


,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,credit_score_segment,age_group,balance_salary_ratio,engagement_status,churn_label
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,Fair,Middle-Aged,0.0000,Active,Churned
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,Fair,Middle-Aged,0.7447,Active,Retained
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,Poor,Middle-Aged,1.4014,Inactive,Churned
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,Good,Middle-Aged,0.0000,Inactive,Retained
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,Exceptional,Middle-Aged,1.5870,Active,Retained


In [3]:
!pip install pymysql

  

  

  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)


In [6]:
import urllib.parse
from sqlalchemy import create_engine
import pandas as pd

# 1. Credentials
username = "root"
raw_password = "Keerthu@2108"
encoded_password = urllib.parse.quote_plus(raw_password)

host = "localhost"
port = 3306
database = "bank_analytics"

# 2. Database Engine with encoded password
db_engine = create_engine(
    f"mysql+pymysql://{username}:{encoded_password}@{host}:{port}/{database}"
)

# 3. Load DataFrame into MySQL
churn.to_sql(
    "bank_churn",
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Data successfully loaded into MySQL table: bank_churn")

# 4. Verify row count
verification = pd.read_sql("SELECT COUNT(*) AS total_rows FROM bank_churn", db_engine)
print(verification)

Data successfully loaded into MySQL table: bank_churn
   total_rows
0       10000


In [4]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = "root"
password = "Keerthu@2108"
host = "localhost"
port = 3306
database = "bank_analytics"

encoded_password = quote_plus(password)

db_engine = create_engine(
    f"mysql+pymysql://{username}:{encoded_password}@{host}:{port}/{database}"
)

In [5]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM bank_churn",
    db_engine
)

,total_rows
0,10000


In [6]:
pd.read_sql(
    "SELECT * FROM bank_churn LIMIT 5",
    db_engine
)

,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,credit_score_segment,age_group,balance_salary_ratio,engagement_status,churn_label
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,Fair,Middle-Aged,0.0000,Active,Churned
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,Fair,Middle-Aged,0.7447,Active,Retained
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,Poor,Middle-Aged,1.4014,Inactive,Churned
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,Good,Middle-Aged,0.0000,Inactive,Retained
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,Exceptional,Middle-Aged,1.5870,Active,Retained
